# D · Trajectory Forecasting: run everything

1. Add the 3 secrets (key icon on the left; switch **Notebook access** on).
2. Click **Runtime → Run all**.

It takes about 40 minutes and costs about $3–4. It stops early with a clear message if something is wrong.

In [ ]:
# Step 1: find the files on Drive, unzip the data, install the tools
from google.colab import drive
drive.mount('/content/drive')
import glob, os, subprocess
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips = glob.glob('/content/drive/MyDrive/**/data_v1_share_v1.2.zip', recursive=True)
assert roots, 'Cannot find the D_trajectory folder in My Drive. Upload it first.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive. Upload it first.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
print('ROOT =', ROOT); print('DATA =', DATA)
!pip -q install openai "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

In [ ]:
# Step 2: read your secrets (Azure: 3 secrets; or OpenAI: just OPENAI_API_KEY)
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
if get('AZURE_OPENAI_ENDPOINT'):
    for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_DEPLOYMENT']:
        assert get(k), f'Secret {k} is missing'
        os.environ[k] = get(k)
    print('Step 2 OK: Azure, deployment =', os.environ['AZURE_OPENAI_DEPLOYMENT'])
else:
    assert get('OPENAI_API_KEY'), 'No secrets found. Add the 3 Azure secrets, or OPENAI_API_KEY.'
    os.environ['OPENAI_API_KEY'] = get('OPENAI_API_KEY')
    os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5-mini'
    print('Step 2 OK: OpenAI API, model = gpt-5-mini')

In [ ]:
# Step 3: one tiny test call to Azure
r = subprocess.run(['python', 'run_trajectory_llm.py', '--root', ROOT, '--data', DATA, '--ping'], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0 and 'connection ok' in r.stdout, 'Azure test failed. Copy the error above and send it to Claude.'
print('Step 3 OK')

In [ ]:
# Step 4: run the model (3 test cases, then 100 cases without A's state, then 100 cases with A's state)
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
runs = [['--split', 'dev', '--workers', '1'],
        ['--split', 'core', '--workers', '2'],
        ['--split', 'core', '--workers', '2', '--state', STATE, '--state-version', 'A_v1']]
for args in runs:
    for attempt in range(3):  # re-run picks up only failed cases (finished ones are saved)
        r = subprocess.run(['python', 'run_trajectory_llm.py', '--root', ROOT, '--data', DATA] + args, capture_output=True, text=True)
        print(r.stdout[-600:], r.stderr[-1500:])
        if r.returncode == 0 and 'failed 0' in r.stdout:
            break
print('Step 4 OK')

In [ ]:
# Step 5: score, plot and build Table 3
for s in ['score_trajectory.py', 'plot_trajectory.py', 'build_table3.py']:
    r = subprocess.run(['python', s, '--root', ROOT], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
import pandas as pd
m = pd.read_csv(f'{ROOT}/metrics/D/trajectory_metrics.csv')
display(m[['method', 'input_setting', 'scored_n', 'failed_n', 'c_index', 'ibs']].round(3))
from IPython.display import Image, display as show
show(Image(f'{ROOT}/results/trajectory_survival_plot.png'))
print('ALL DONE. Take a screenshot of this table and send it to Claude.')